# Course 1, Week 2 — The PyTorch Workflow

- [DeepLearning.AI platform](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson)
- [Week notes](README.md)
- [GitHub issue #2](https://github.com/majorgilles/pytorch_for_deep_learning/issues/2)

**Focus:** Build the training loop: model, loss, optimizer, gradients, evaluation, and saving.


## From tensor fundamentals to image classification

The previous week introduced tensors and the basic PyTorch training loop. This week applies that foundation to a more demanding task: recognizing handwritten characters from images. A character classifier predicts one class for each segmented image; those predictions can then be assembled into readable text.

### Why images require a stronger workflow

An image contains many more input values than the small tabular examples used previously. A grayscale MNIST image has height $28$ and width $28$, giving $28 \times 28 = 784$ pixel values. A batch is commonly represented with shape $[B, C, H, W]$, where:

- $B$ is the number of images in the batch.
- $C=1$ is the grayscale channel.
- $H=W=28$ are the image dimensions.

The classifier must map each image to one of ten digit classes, $0$ through $9$. MNIST is a useful starting point because its images are consistently sized, centered, labeled, and small enough to train quickly.

### Learning path

The image-classification workflow introduces several tools that become essential as datasets and models grow:

1. Load and batch image data efficiently.
2. Build multi-layer models with more flexibility than a simple linear predictor.
3. Choose losses and optimizers suited to classification.
4. Inspect the training process through predictions, losses, and gradients.
5. Move tensors and models between CPU and GPU devices safely.

MNIST provides a compact environment for practicing this complete pipeline before progressing to convolutional neural networks and more varied handwriting.


## Loading data the PyTorch way

The machine learning pipeline still begins with data, but image datasets make memory management more important. Loading every sample into RAM may work for a small delivery table, but it stops scaling as the number and size of the records grow. The practical rule is simple: **load only the data needed for the current batch**.

PyTorch organizes this work with three tools that fit together:

1. **Transforms** prepare each sample as it is loaded.
2. **Dataset** describes how to find, load, and count individual samples.
3. **DataLoader** requests samples from the dataset and serves them in batches.

The resulting flow is:

$$\text{stored sample} \longrightarrow \text{transform} \longrightarrow \text{dataset} \longrightarrow \text{batch} \longrightarrow \text{model}.$$

### 1. Transform each sample

`transforms.Compose` applies several operations in sequence. For common image inputs, `ToTensor` converts the image to a tensor and scales unsigned 8-bit pixel values from $[0,255]$ to $[0,1]$. `Normalize` then transforms each channel using

$$x' = \frac{x-\mu}{\sigma},$$

which can give optimization a better-scaled input. More advanced augmentation and preprocessing techniques come later.

### 2. Address samples through a dataset

A `Dataset` provides two essential behaviors:

- `len(dataset)` reports the number of samples.
- `dataset[index]` loads one sample and its label.

The sample can be read from storage and transformed only when requested rather than being preloaded with the entire dataset. Built-in datasets such as MNIST also support selecting the training or test split and downloading missing files. Custom dataset classes use the same interface.

### 3. Serve batches with a data loader

A `DataLoader` groups requested samples into manageable batches. `batch_size` controls how many examples are returned at once, while `shuffle=True` changes the training order between passes through the dataset. For MNIST, a batch of 64 images has shape $[64,1,28,28]$ and its labels have shape $[64]$.


In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Apply each preprocessing step when an image is requested.
image_transform: transforms.Compose = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5,), std=(0.5,)),
    ]
)

# The dataset identifies samples; it does not place every image in one tensor.
train_dataset: datasets.MNIST = datasets.MNIST(
    root="data", train=True, download=True, transform=image_transform
)

# The loader retrieves one shuffled batch at a time.
train_loader: DataLoader = DataLoader(
    train_dataset, batch_size=64, shuffle=True
)

batch: tuple[torch.Tensor, torch.Tensor] = next(iter(train_loader))
images, labels = batch
print(f"image batch: {images.shape}")
print(f"label batch: {labels.shape}")

image batch: torch.Size([64, 1, 28, 28])
label batch: torch.Size([64])


This pattern scales from delivery records to image collections: transform one sample, let the dataset retrieve it, and let the data loader assemble only the next batch. With the data pipeline in place, the next step is to examine how losses, gradients, and optimizers train the model.


## Building models with `nn.Module`

`nn.Sequential` is convenient when data passes through layers in one fixed order. A custom `nn.Module` expresses the same computation with more control and is the standard pattern for models with branches, skip connections, or other custom behavior.

Every custom module has two central parts:

1. **`__init__` defines the layers.** Calling `super().__init__()` first lets PyTorch register their learnable weights and biases.
2. **`forward` defines the data flow.** It describes the order in which those layers process an input tensor.

Call the module as `model(inputs)`, not `model.forward(inputs)`. The module call invokes `forward` while preserving PyTorch's hooks and other internal bookkeeping.


In [2]:
from torch import nn, optim


class DigitClassifier(nn.Module):
    """Map image batches of shape $[B, 1, 28, 28]$ to logits of shape $[B, 10]$."""

    def __init__(self) -> None:
        super().__init__()
        self.flatten: nn.Flatten = nn.Flatten()
        self.hidden: nn.Linear = nn.Linear(28 * 28, 128)
        self.activation: nn.ReLU = nn.ReLU()
        self.output: nn.Linear = nn.Linear(128, 10)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """Return class logits for inputs with shape $[B, 1, 28, 28]$."""
        flattened: torch.Tensor = self.flatten(inputs)
        hidden: torch.Tensor = self.activation(self.hidden(flattened))
        return self.output(hidden)


model: DigitClassifier = DigitClassifier()
logits: torch.Tensor = model(images)
print(f"input: {images.shape} -> logits: {logits.shape}")

input: torch.Size([64, 1, 28, 28]) -> logits: torch.Size([64, 10])


## Training in the correct order

A standard PyTorch training step follows the same sequence for every batch:

1. `optimizer.zero_grad()` clears gradients accumulated by earlier batches.
2. `model(inputs)` performs the forward pass.
3. The loss function compares the logits with the correct labels.
4. `loss.backward()` computes parameter gradients for the current batch.
5. `optimizer.step()` updates the parameters using those gradients.

The order matters even when incorrect code does not immediately raise an exception. Stepping before backpropagation uses stale gradients, clearing gradients after backpropagation discards the new gradients, and clearing only once outside the loop causes gradients from multiple batches to accumulate.


In [3]:
model.train()
loss_function: nn.CrossEntropyLoss = nn.CrossEntropyLoss()
optimizer: optim.SGD = optim.SGD(model.parameters(), lr=0.01)

optimizer.zero_grad()
logits = model(images)
loss: torch.Tensor = loss_function(logits, labels)
loss.backward()
optimizer.step()

print(f"training loss: {loss.item():.4f}")

training loss: 2.3162


## Evaluating on unseen data

Evaluation checks whether the model generalizes beyond its training examples. Two PyTorch tools prepare the model for this phase:

- `model.eval()` switches layers with training-specific behavior, such as dropout and batch normalization, into evaluation mode. It does **not** calculate a score.
- `torch.no_grad()` disables gradient tracking, reducing unnecessary memory and computation.

For classification, accuracy is the fraction of correct predictions:

$$\operatorname{accuracy} = \frac{\text{correct predictions}}{\text{total predictions}}.$$

Predictions must be evaluated on a separate test or validation split. Measuring only the training data cannot reveal whether the model learned a reusable pattern or merely fit examples it already saw. Call `model.train()` again before resuming training.


In [4]:
test_dataset: datasets.MNIST = datasets.MNIST(
    root="data", train=False, download=True, transform=image_transform
)
test_loader: DataLoader = DataLoader(
    test_dataset, batch_size=64, shuffle=False
)
test_batch: tuple[torch.Tensor, torch.Tensor] = next(iter(test_loader))
test_images: torch.Tensor = test_batch[0]
test_labels: torch.Tensor = test_batch[1]

model.eval()
with torch.no_grad():
    test_logits: torch.Tensor = model(test_images)
    predictions: torch.Tensor = test_logits.argmax(dim=1)

correct: int = int((predictions == test_labels).sum().item())
total: int = test_labels.numel()
accuracy: float = correct / total
print(f"accuracy on one unseen batch: {accuracy:.1%}")

# Restore training behavior before any further optimization.
model.train()

accuracy on one unseen batch: 9.4%


DigitClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (hidden): Linear(in_features=784, out_features=128, bias=True)
  (activation): ReLU()
  (output): Linear(in_features=128, out_features=10, bias=True)
)